### **Dynamic Programming**
#### **Frozen Lake environment**

Solution of Frozen Lake using dynamic Programming
It uses an unwrapped environment. See description of unwrapped in the exercise definition

In [1]:
import gymnasium as gym
import time
import numpy as np
import matplotlib.pyplot as plt
import session_info
from IPython.display import clear_output, display

In [2]:
# Initialize the FrozenLake environment
env = gym.make('FrozenLake-v1', is_slippery=False, render_mode='human')
env = env.unwrapped  # Access raw environment (no TimeLimit wrapper)
env.reset()          # Reset the environment

# Parameters
gamma = 0.99  # Discount factor for future rewards
theta = 1e-6  # Convergence threshold for value iteration

# Evaluates a given policy by calculating its value function using an iterative approach
def policy_evaluation(env, policy, theta=theta, gamma=gamma):
    """
    Evaluate a given policy by computing its value function.
    Args:
        env: The FrozenLake environment.
        policy: The policy to evaluate.
        theta: Convergence threshold for policy evaluation.
        gamma: Discount factor.
    Returns:
        V: The value function for the given policy.
    """
    V = np.zeros(env.observation_space.n)  # Initialize value function

    while True:
        delta = 0  # Track maximum value change across states
        for s in range(env.observation_space.n):
            old_v = V[s]  # Store current value of state
            new_v = 0  # Calculate new value

            # Sum expected values for each action based on policy
            for a, action_prob in enumerate(policy[s]):
                for prob, next_state, reward, done in env.P[s][a]:
                    future_value = 0 if done else gamma * V[next_state]
                    new_v += action_prob * prob * (reward + future_value)

            V[s] = new_v  # Update value function
            delta = max(delta, np.abs(old_v - new_v))

        if delta < theta:  # Check for convergence
            break

    return V

In [3]:
# Improves a given policy using the current value function estimates
def policy_improvement(env, V, gamma=gamma):
    """
    Improve the policy based on a given value function.
    Args:
        env: The FrozenLake environment.
        V: Value function for the policy.
        gamma: Discount factor.
    Returns:
        policy: The improved policy.
    """
    policy = np.zeros([env.observation_space.n, env.action_space.n])  # Initialize policy

    for s in range(env.observation_space.n):
        q_values = np.zeros(env.action_space.n)  # Calculate Q-values for all actions
        for a in range(env.action_space.n):
            for prob, next_state, reward, done in env.P[s][a]:
                future_value = 0 if done else gamma * V[next_state]
                q_values[a] += prob * (reward + future_value)

        best_actions = np.where(q_values == np.max(q_values))[0]  # Select optimal actions
        policy[s, best_actions] = 1.0 / len(best_actions)  # Distribute probability evenly

    return policy

In [4]:
# Computes an optimal policy using policy iteration
def policy_iteration(env, theta=theta, gamma=gamma, max_iterations=1000):
    """
    Perform policy iteration to compute the optimal policy.
    Args:
        env: The FrozenLake environment.
        theta: Convergence threshold.
        gamma: Discount factor.
        max_iterations: Maximum number of iterations.
    Returns:
        policy: Optimal policy.
        V: Value function for the optimal policy.
    """
    policy = np.ones([env.observation_space.n, env.action_space.n]) / env.action_space.n  # Initialize policy
    iterations = 0

    while iterations < max_iterations:
        V = policy_evaluation(env, policy, theta, gamma)
        new_policy = policy_improvement(env, V, gamma)

        if np.all(np.abs(policy - new_policy) < theta):  # Check for policy stability
            break

        policy = new_policy
        iterations += 1

    print(f'Policy iteration converged after {iterations} iterations')
    return policy, V

In [5]:
# Displays the policy for the FrozenLake environment in a grid format
def print_policy(env, policy):
    """
    Print the policy in a human-readable format for a 4x4 FrozenLake grid.
    """
    print("\nPolicy Format (4x4):")
    print("-" * 50)
    actions = ['←', '↓', '→', '↑']  # Define action symbols

    holes = [(i, j) for i in range(env.nrow) for j in range(env.ncol) if env.desc[i][j] == b'H']  # Identify holes

    for i in range(env.nrow):
        for j in range(env.ncol):
            s = i * env.ncol + j
            if (i, j) in holes:
                print('H', end=' ')  # Mark holes
            elif (i, j) == (3, 3):
                print('G', end=' ')  # Mark goal
            elif (i, j) == (0, 0):
                print('S', end=' ')  # Mark start
            else:
                a = np.argmax(policy[s])  # Select action with highest probability
                print(actions[a], end=' ')
        print()

In [6]:
# Displays the value function for the FrozenLake environment in a grid format
def print_value_function(V, env):
    """
    Print the value function in a human-readable format for a 4x4 grid.
    """
    print("\nValue Function Format (4x4):")
    print("-" * 50)
    holes = [(i, j) for i in range(env.nrow) for j in range(env.ncol) if env.desc[i][j] == b'H']  # Identify holes

    for i in range(env.nrow):
        for j in range(env.ncol):
            s = i * env.ncol + j
            if (i, j) in holes:
                print('H'.center(6), end=' ')  # Mark holes
            elif (i, j) == (3, 3):
                print('G'.center(6), end=' ')  # Mark goal
            elif (i, j) == (0, 0):
                print('S'.center(6), end=' ')  # Mark start
            else:
                print(f'{V[s]:5.3f}'.center(6), end=' ')
        print()

In [7]:
# Computes the optimal policy using value iteration 
def value_iteration(env, theta=theta, gamma=gamma, max_iterations=1000):
    """
    Perform value iteration to compute the optimal policy.
    Args:
        env: The FrozenLake environment.
        theta: Convergence threshold.
        gamma: Discount factor.
        max_iterations: Maximum number of iterations.
    Returns:
        policy: Optimal policy.
        V: Optimal value function.
    """
    V = np.zeros(env.observation_space.n)  # Initialize value function
    iterations = 0

    while iterations < max_iterations:
        delta = 0
        for s in range(env.observation_space.n):
            old_v = V[s]
            q_values = np.zeros(env.action_space.n)
            for a in range(env.action_space.n):
                for prob, next_state, reward, done in env.P[s][a]:
                    future_value = 0 if done else gamma * V[next_state]
                    q_values[a] += prob * (reward + future_value)

            V[s] = np.max(q_values)  # Update value function
            delta = max(delta, np.abs(old_v - V[s]))

        if delta < theta:  # Check for convergence
            break
        iterations += 1

    print(f'Value iteration converged after {iterations} iterations')
    policy = policy_improvement(env, V, gamma)
    return policy, V


In [8]:
# Define the function to simulate the optimal policy
def simulate_optimal_policy(env, policy, max_steps=100):
    """
    Simulate an episode using the optimal policy and display each step.
    Args:
        env: The FrozenLake environment.
        policy: The optimal policy to follow.
        max_steps: Maximum number of steps in an episode.
    Returns:
        total_reward: Total reward accumulated during the episode.
        path: Sequence of states visited.
    """
    state, _ = env.reset()
    total_reward = 0
    path = [state]

    for _ in range(max_steps):
        action = np.argmax(policy[state])  # Choose optimal action
        next_state, reward, done, _, _ = env.step(action)
        total_reward += reward
        path.append(next_state)
        state = next_state

        # Display the current state
        frame = env.render()
#        plt.imshow(frame)
#        plt.axis('off')
#        display(plt.gcf())
#        clear_output(wait=True)

        if done:
            break

    return total_reward, path

# Run policy iteration
pi_policy, pi_value = policy_iteration(env, theta=theta, gamma=gamma)
print("\nPolicy from Policy Iteration:")
print_policy(env, pi_policy)

# Run value iteration
vi_policy, vi_value = value_iteration(env, theta=theta, gamma=gamma)
print("\nPolicy from Value Iteration:")
print_policy(env, vi_policy)
print_value_function(vi_value, env)

Policy iteration converged after 2 iterations

Policy from Policy Iteration:

Policy Format (4x4):
--------------------------------------------------
S → ↓ ← 
↓ H ↓ H 
→ ↓ ↓ H 
H → → G 
Value iteration converged after 6 iterations

Policy from Value Iteration:

Policy Format (4x4):
--------------------------------------------------
S → ↓ ← 
↓ H ↓ H 
→ ↓ ↓ H 
H → → G 

Value Function Format (4x4):
--------------------------------------------------
  S    0.961  0.970  0.961  
0.961    H    0.980    H    
0.970  0.980  0.990    H    
  H    0.990  1.000    G    


In [9]:
def evaluate_gammas(env, gamma_values, theta=1e-6, max_iterations=1000):
    """
    Evaluate the performance of policy and value iteration for different gamma values.

    Args:
        env: The FrozenLake environment.
        gamma_values: List of gamma (discount factor) values to test.
        theta: Convergence threshold for policy and value iteration.
        max_iterations: Maximum number of iterations for each method.

    Returns:
        results: List of dictionaries containing evaluation results for each gamma.
    """
    results = []
    
    for gamma in gamma_values:
        print(f"\nEvaluating for gamma: {gamma:.2f}")

        # Run policy iteration for the current gamma
        pi_policy, pi_value = policy_iteration(env, theta=theta, gamma=gamma, max_iterations=max_iterations)
        print(f"\nPolicy from Policy Iteration for gamma={gamma:.2f}:")
        print_policy(env, pi_policy)

        # Run value iteration for the current gamma
        vi_policy, vi_value = value_iteration(env, theta=theta, gamma=gamma, max_iterations=max_iterations)
        print(f"\nPolicy from Value Iteration for gamma={gamma:.2f}:")
        print_policy(env, vi_policy)
        print_value_function(vi_value, env)

        # Record the results for comparison
        results.append({
            'gamma': gamma,
            'pi_policy': pi_policy,
            'pi_value': pi_value,
            'vi_policy': vi_policy,
            'vi_value': vi_value
        })

    return results

# Define a list of gamma values to test
gamma_values = [0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

# Evaluate different gamma values
results = evaluate_gammas(env, gamma_values)


Evaluating for gamma: 0.50
Policy iteration converged after 2 iterations

Policy from Policy Iteration for gamma=0.50:

Policy Format (4x4):
--------------------------------------------------
S → ↓ ← 
↓ H ↓ H 
→ ↓ ↓ H 
H → → G 
Value iteration converged after 6 iterations

Policy from Value Iteration for gamma=0.50:

Policy Format (4x4):
--------------------------------------------------
S → ↓ ← 
↓ H ↓ H 
→ ↓ ↓ H 
H → → G 

Value Function Format (4x4):
--------------------------------------------------
  S    0.062  0.125  0.062  
0.062    H    0.250    H    
0.125  0.250  0.500    H    
  H    0.500  1.000    G    

Evaluating for gamma: 0.70
Policy iteration converged after 2 iterations

Policy from Policy Iteration for gamma=0.70:

Policy Format (4x4):
--------------------------------------------------
S → ↓ ← 
↓ H ↓ H 
→ ↓ ↓ H 
H → → G 
Value iteration converged after 6 iterations

Policy from Value Iteration for gamma=0.70:

Policy Format (4x4):
----------------------------------

### Analysis of Gamma Values

The evaluation of different gamma values in the FrozenLake environment demonstrated interesting patterns and insights. <br>
For gamma values ranging from 0.50 to 0.99, both policy iteration and value iteration methods showed quick convergence, with policy iteration consistently converging after just 2 iterations, while value iteration stabilized after 6 iterations. The derived policies across these gamma values were consistent, indicating stability in the optimal policy determined by both methods. As the gamma value increased, the value function for each state also increased, highlighting a greater emphasis on future rewards. <br>
<br>
However, when evaluating with a gamma value of 1.00, policy iteration exhibited a notable change, taking the maximum of 1000 iterations to converge. This suggests that treating immediate and future rewards equally introduces complexity, leading to prolonged policy evaluation. Despite this, value iteration maintained consistent convergence at 6 iterations. <br>
<br>
The optimal policy derived at gamma = 1.00 differed from those at lower gamma values, reflecting a strong influence on decision-making when future rewards are given equal priority to immediate ones. Overall, this evaluation underscores how varying the discount factor influences policy stability and value functions, with higher gamma values accentuating long-term reward considerations and potentially leading to more complex policy solutions.

In [10]:
# Simulate using the optimal policy from value iteration
total_reward, path = simulate_optimal_policy(env, vi_policy, max_steps=100)
print("\nTotal reward:", total_reward)
print("Path taken:", path)


Total reward: 1.0
Path taken: [0, 4, 8, 9, 13, 14, 15]


In [11]:
session_info.show(html=False)

-----
gymnasium           1.0.0
matplotlib          3.9.1
numpy               1.26.4
session_info        1.0.0
-----
IPython             8.26.0
jupyter_client      8.6.2
jupyter_core        5.7.2
-----
Python 3.12.3 (main, Feb  4 2025, 14:48:35) [GCC 13.3.0]
Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
-----
Session information updated at 2025-05-21 14:06
